# Patch: Fix CDL Cropland Features

## What this notebook does

Fixes a column name formatting bug in the CDL (Cropland Data Layer) extraction that caused two problems in `joint_county_year_2018_2019.csv`:

1. **`pct_cropland` = 100% for all counties** — should range from ~0–92%
2. **`corn_acres`, `soybean_acres`, `cotton_acres`, `wheat_acres`, `hay_acres`, `fruit_veg_acres`, `rice_acres`, `sorghum_acres` = 0 for all counties**

## Root cause

The CDL API returns CSV data with leading whitespace in column names (`' Category'`, `'  Acreage'` instead of `'Category'`, `'Acreage'`). The column detection logic in `build_joint_dataset.ipynb` looked for exact matches (`'Category'`, `'Acres'`, etc.) and silently fell back to wrong columns when those didn't match. As a result:
- The NON_CROP filter (forests, wetlands, developed land etc.) never matched anything
- `cropland_acres` ended up equalling `total_area_acres` for every county
- All specific crop acre columns summed to zero

## What is NOT affected
- All pesticide columns (total, by class, by compound)
- All demographic columns (census, NCHS urban-rural)
- All health outcome columns (CASTHMA, COPD, CSMOKING, OBESITY, DIABETES)
- `cropland_diversity` and `county_crop_concentration` (computed differently, unaffected)

## Requirements
- `cdl_2019_progress.csv` in `data/` — the raw CDL API pull (already in repo)
- `joint_county_year_2018_2019.csv` in `data/` — the joint dataset to patch

## Output
- Overwrites broken columns in `joint_county_year_2018_2019.csv` with corrected values
- Saves a backup of the original as `joint_county_year_2018_2019_pre_cropland_fix.csv`

In [ ]:
import pandas as pd
import numpy as np
import re
import os

DATA_DIR = "../data"
CDL_RAW_PATH      = os.path.join(DATA_DIR, "cdl_2019_progress.csv")
JOINT_PATH        = os.path.join(DATA_DIR, "joint_county_year_2018_2019.csv")
BACKUP_PATH       = os.path.join(DATA_DIR, "joint_county_year_2018_2019_pre_cropland_fix.csv")
OUTPUT_PATH       = JOINT_PATH  # overwrite in place

print("Paths configured.")
print(f"  CDL raw:  {CDL_RAW_PATH}")
print(f"  Joint:    {JOINT_PATH}")
print(f"  Backup:   {BACKUP_PATH}")

## 1. Verify the bug exists before patching

In [ ]:
joint_orig = pd.read_csv(JOINT_PATH)
print("Joint dataset shape:", joint_orig.shape)

print("\n--- Confirming bug ---")
print("pct_cropland unique values (should be 1 if bugged):", 
      joint_orig['pct_cropland'].nunique())
print("pct_cropland mean (should be 100.0 if bugged):", 
      joint_orig['pct_cropland'].mean().round(2))
print("corn_acres mean (should be 0.0 if bugged):", 
      joint_orig['corn_acres'].mean().round(2))
print("cropland_acres == total_area_acres for all rows:", 
      (joint_orig['cropland_acres'] == joint_orig['total_area_acres']).mean().round(4))

if joint_orig['pct_cropland'].nunique() == 1:
    print("\n✓ Bug confirmed. Proceeding with patch.")
else:
    print("\n! pct_cropland has variance — bug may already be fixed. "
          "Check before proceeding.")

## 2. Load and clean the raw CDL data

In [ ]:
cdl = pd.read_csv(CDL_RAW_PATH)

# THE FIX: strip leading/trailing whitespace from column names and category values
cdl.columns = cdl.columns.str.strip()
cdl['Category'] = cdl['Category'].str.strip()

# Standardize FIPS to 5-digit string
cdl['county_fips'] = cdl['county_fips'].astype(str).str.zfill(5)

print("CDL shape:", cdl.shape)
print("Columns:", list(cdl.columns))
print("Unique counties:", cdl['county_fips'].nunique())
print("\nSample categories (first 10):")
print(cdl['Category'].unique()[:10])

## 3. Recompute cropland features correctly

In [ ]:
# Non-crop land cover categories to EXCLUDE from cropland total
NON_CROP = [
    "developed", "open water", "water", "forest",
    "wetland", "barren", "shrubland", "grassland", "pasture"
]

# Crop-specific acreage groups
CROP_GROUPS = {
    "corn_acres":       ["corn", "sweet corn", "maize"],
    "soybean_acres":    ["soybean", "soybeans", "soy"],
    "cotton_acres":     ["cotton"],
    "wheat_acres":      ["wheat", "durum", "spring wheat", "winter wheat"],
    "hay_acres":        ["hay", "alfalfa"],
    "fruit_veg_acres":  [
        "orchard", "vineyard", "fruit", "vegetable", "grape",
        "apple", "citrus", "berry", "tomato", "peach", "almond",
        "potato", "melon", "lettuce", "onion", "carrot",
        "broccoli", "pepper", "strawberry"
    ],
    "rice_acres":       ["rice"],
    "sorghum_acres":    ["sorghum"],
}

# Precompute lowercase category name
cdl['_name'] = cdl['Category'].str.lower()
cdl['_is_noncrop'] = cdl['_name'].str.contains(
    "|".join(NON_CROP), case=False, na=False
)

print("Non-crop categories identified:")
print(cdl[cdl['_is_noncrop']]['Category'].value_counts().head(15))

In [ ]:
results = []

for fips, group in cdl.groupby('county_fips'):
    total_area    = group['Acreage'].sum()
    crop_rows     = group[~group['_is_noncrop']]
    cropland_acres = crop_rows['Acreage'].sum()
    pct_cropland  = 100 * cropland_acres / total_area if total_area > 0 else 0

    row = {
        'FIPS':               fips,
        'cropland_acres':     cropland_acres,
        'total_area_acres':   total_area,
        'pct_cropland':       pct_cropland,
    }

    for feat, keywords in CROP_GROUPS.items():
        mask = group['_name'].str.contains(
            "|".join(re.escape(k) for k in keywords), case=False, na=False
        )
        row[feat] = group.loc[mask, 'Acreage'].sum()

    results.append(row)

fixed = pd.DataFrame(results)

print("Fixed cropland shape:", fixed.shape)
print("\npct_cropland (fixed):")
print(fixed['pct_cropland'].describe().round(1))
print("\ncorn_acres (fixed):")
print(fixed['corn_acres'].describe().round(1))

## 4. Sanity check fixed values on a known county

In [ ]:
# County 01001 (Autauga County, AL) — we can verify against the raw CDL screenshot
# Expected: pct_cropland ~13.3%, cotton ~11,046 acres, deciduous forest ~73,611 acres excluded
sample = fixed[fixed['FIPS'] == '01001'].iloc[0]
print("County 01001 sanity check:")
for col in ['cropland_acres', 'total_area_acres', 'pct_cropland',
            'corn_acres', 'cotton_acres', 'soybean_acres']:
    print(f"  {col}: {sample[col]:.1f}")

print("\nExpected pct_cropland ~13.3% ✓" 
      if abs(sample['pct_cropland'] - 13.3) < 1 
      else "\n✗ Unexpected value — check raw data")
print("Expected cotton_acres ~11,046 ✓" 
      if abs(sample['cotton_acres'] - 11046) < 100 
      else "\n✗ Unexpected value — check raw data")

## 5. Back up original and save patched joint dataset

In [ ]:
# Save backup of original
joint_orig.to_csv(BACKUP_PATH, index=False)
print(f"Backup saved: {BACKUP_PATH}")

# Drop broken columns from joint dataset
BROKEN_COLS = [
    'cropland_acres', 'total_area_acres', 'pct_cropland',
    'corn_acres', 'soybean_acres', 'cotton_acres', 'wheat_acres',
    'hay_acres', 'fruit_veg_acres', 'rice_acres', 'sorghum_acres'
]
joint_patched = joint_orig.drop(columns=BROKEN_COLS)

# Standardize FIPS for merge
joint_patched['FIPS'] = joint_patched['FIPS'].astype(str).str.zfill(5)

# Merge fixed cropland columns (county-level, same value for 2018 and 2019)
joint_patched = joint_patched.merge(fixed, on='FIPS', how='left')

print(f"\nJoint shape before patch: {joint_orig.shape}")
print(f"Joint shape after patch:  {joint_patched.shape}")
print(f"\nColumn count same: {joint_orig.shape[1] == joint_patched.shape[1]}")

## 6. Verify the patch worked

In [ ]:
print("--- Verifying patch ---")
print("pct_cropland unique values (should be >> 1 now):",
      joint_patched['pct_cropland'].nunique())
print("pct_cropland mean (should be ~25, not 100):",
      joint_patched['pct_cropland'].mean().round(1))
print("pct_cropland range: ",
      joint_patched['pct_cropland'].min().round(1), "–",
      joint_patched['pct_cropland'].max().round(1))
print("corn_acres mean (should be ~30k, not 0):",
      joint_patched['corn_acres'].mean().round(0))
print("corn_acres non-zero rows:",
      (joint_patched['corn_acres'] > 0).sum())

# Confirm unaffected columns unchanged
print("\n--- Confirming unaffected columns unchanged ---")
for col in ['pesticide_total_kg', 'CASTHMA', 'COPD', 'median_income', 'cropland_diversity']:
    orig_mean = joint_orig[col].mean()
    new_mean  = joint_patched[col].mean()
    match = "✓" if abs(orig_mean - new_mean) < 0.001 else "✗ CHANGED"
    print(f"  {col}: {match}")

## 7. Save patched dataset

In [ ]:
joint_patched.to_csv(OUTPUT_PATH, index=False)
print(f"Patched dataset saved to: {OUTPUT_PATH}")
print(f"Shape: {joint_patched.shape}")
print("\nNote: original backed up at:", BACKUP_PATH)
print("\nThe following columns are now fixed:")
for col in BROKEN_COLS:
    print(f"  {col}")
print("\nNo other columns were changed.")
print("\nTrain/test split notebooks do NOT need to be rerun — "
      "splits are based on FIPS and CASTHMA/COPD stratification, "
      "neither of which changed.")
print("Modeling notebooks that used pct_cropland or specific crop "
      "acre columns as features should be rerun.")